In [7]:
import os
import numpy as np
from torch.utils.data import Dataset
from torchvision import transforms
from torchvision.datasets import ImageFolder
from PIL import Image
from torch import nn, optim
from cv2 import reduce
import torch
from torch.utils.data import Dataset, DataLoader
import gc

class CustomImageFolder(Dataset):
    def __init__(self, root_dir):
        self.root_dir = root_dir
        labels = []
        images = []
        for label_folder in os.listdir(root_dir):
            full_label_folder = os.path.join(root_dir, label_folder)
            label = int(label_folder)
            for sample_folder in os.listdir(full_label_folder):
                full_sample_folder = os.path.join(full_label_folder, sample_folder)
                image_list = []
                for image_file in os.listdir(full_sample_folder):
                    full_image_file = os.path.join(full_sample_folder, image_file)
                    image = Image.open(full_image_file)
                    image_list.append(image)
                images.append(image_list)
                labels.append(label)

        self.labels = labels
        self.images = images
        self.classes = np.unique(labels)
        self.transform = transforms.Compose([
          transforms.GaussianBlur(9, sigma=(0.1, 2.0)),
          transforms.Resize(256,interpolation=transforms.InterpolationMode.BICUBIC),
          transforms.CenterCrop(224),
          transforms.ToTensor(),
          transforms.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ])

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        # return torch.tensor(self.images[idx]).float(), torch.tensor(self.labels[idx]).long()
        # Apply transformation if provided
        if self.transform:
            image = self.transform(self.images[idx])
        return image, self.labels[idx]  # Returning image and its path

In [3]:
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
device

device(type='cuda')

In [8]:


class MLP(nn.Module):
    
    def __init__(self, dim, inner_dim,n_class,encoder):     #dim would be the output image feature from dinov2                                
        super().__init__()
        # mlp with GELU activation function
        self.encoder = encoder
        self.mlp = nn.Sequential(
            nn.Linear(dim, n_class),
        )

    def forward(self, x):
        # x is [16,8,3,224,224]
        avg = []
        
        for i in range(8):
            xi = x[:,i,:]
            #encode x to [8,384]
            with torch.no_grad():
                e = self.encoder(xi).reshape(x.shape[0],1,384)
            avg.append(e)
        avg = torch.cat(avg,dim=1)    
        avg = reduce(avg, "f t c -> f c",'mean')        #[16,384]
        return self.mlp(avg)


In [11]:

# data loading params
dataset_dir = "/media/osero/SamsungSSD/CMPE_SSD/frame-hand_left-c256_TOY"
batch_size = 256
test_batch_size = 1
num_workers = 8
pin_memory = True
num_classes=50
epochs = 10

In [10]:
# Dataset
train_val_data = CustomImageFolder( dataset_dir)

train_len=int(0.85*len(train_val_data))
train_val_split = [ train_len, len(train_val_data) - train_len ] 

train_data , val_data = torch.utils.data.random_split(train_val_data,train_val_split)
test_data = CustomImageFolder( dataset_dir)

# Dataloaders
train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_data, batch_size=test_batch_size)
test_loader = DataLoader(test_data, batch_size=test_batch_size)

In [12]:
dinov2_vits14 = torch.hub.load('facebookresearch/dinov2', 'dinov2_vits14')
dinov2_vits14.to(device)
for param in dinov2_vits14.parameters():
    param.requires_grad= False
model = MLP(384,512,101,dinov2_vits14)
#frames, _ = next(iter(train_loader))
#tb_writer.add_graph(model, frames)
model.to(device)
# define the loss and optimizers
loss_criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-6)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=1, gamma=0.95)

Using cache found in /home/osero/.cache/torch/hub/facebookresearch_dinov2_main


In [14]:

# training step for every epoch
import tqdm


def train_step(loader,epoch,):
    
    model.train()
    total_epoch_loss=0
    
    for batch_id, (video_data,labels) in enumerate(loader):

        # video_data,labels = video_data.to(device), labels.to(device)
        video_data,labels = video_data.to(device), labels.to(device)
        
        optimizer.zero_grad()
        
        prediction = model(video_data)

        loss = loss_criterion(prediction,labels)
        total_epoch_loss += loss.item()

        loss.backward()
        
        optimizer.step()

        del video_data
        del labels

        gc.collect()
        
        #tb_writer.add_scalar("Train/Loss",loss.item(),((len(loader))*(epoch-1))+batch_id)
 
        print(f"\n[Train Epoch]: {epoch} Train Loss: {loss.item()}")
    return total_epoch_loss


# validation step for every epoch
def val_step(loader,epoch=None):

    model.eval()
    total_loss=0
    corrects=0
    
    with torch.no_grad():
        for batch_id, (video_data,labels) in enumerate(loader):

            video_data,labels = (video_data).to(device), labels.to(device)

            prediction = model(video_data)
            
            loss = loss_criterion(prediction,labels)
            total_loss += loss.item()
            corrects+= (torch.argmax(prediction,dim=1)==labels).sum()
    
    accuracy = corrects/(len(loader)*batch_size)
    
    print(f"\n[Val Epoch]: {epoch} , Accuracy: {accuracy}, Valid Loss: {loss.item()}")


    return accuracy

# Driving train test loop
for epoch in range(1,epochs+1):
    train_step(train_loader, epoch)
    val_step(val_loader, epoch)
    scheduler.step()
    torch.save(model,"dino_model.pt")


TypeError: img should be PIL Image or Tensor. Got <class 'list'>